# Tutorial 02 — Rate-½ Convolutional Code and BCJR Decoder

The outer code in MS-PRS is a rate-½ K=3 convolutional code with generators (5, 7) in octal. This tutorial walks through encoding, then running the BCJR (MAP) decoder over an AWGN channel to recover the source bits.

The BCJR algorithm computes the *a posteriori* probability of each bit by a forward (α) and backward (β) recursion over the trellis. Its output is the extrinsic LLR `L_post − L_a`, ready to feed back to the inner equalizer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
from nsm.codec.conv import precompute, encode, decode
from nsm.modem.ask2 import modulate as bpsk_mod, demodulate as bpsk_demod
from nsm.channel.awgn import setup, transmit
rng = np.random.default_rng(0)

## Trellis at a glance

In [ ]:
CODER = {'K': 3, 'octal_code': (0o5, 0o7)}
info  = precompute(CODER, packet_length=200)
print('memory states :', info['total_states'])
print('rate          :', info['rate'])
print('coding length :', info['coding_length'])

## Encode → BPSK → AWGN → BCJR

In [ ]:
src      = rng.integers(0, 2, 200).astype(np.int32)
tailed   = np.concatenate([src, np.zeros(info['memory'], dtype=np.int32)])
coded    = encode(tailed, info['coding_length'], info['polynomials'])
ch       = setup((6, 6, 1), avg_bit_energy=0.5, rate=0.5)
rx       = transmit(bpsk_mod(coded, 0.5), ch['noise_std'][0])
L_chan   = bpsk_demod(rx, ch['noise_var'][0], 0.5)
_, est   = decode(L_chan.astype(np.float32), info['coding_length'], 200,
                  info['n_outputs'], info['memory'], info['total_states'],
                  info['next_states'], info['outputs'])
print('bit errors (200 bits, 6 dB):', int(np.sum(est != src)))

## BER curve vs the union bound
The dominant union-bound term for the (5,7) code uses free distance `d_free = 5`: `BER ≤ ½ erfc(√(d_free·rate·Eb/N0))`.

In [ ]:
from nsm.modem.ask2 import ber_coded_union_bound
PKT = 500
info500 = precompute(CODER, PKT)
eb_no_db = np.arange(0, 8.1, 1.0); ber = []
for snr_db in eb_no_db:
    ch = setup((snr_db, snr_db, 1), 0.5, rate=0.5)
    errs, bits = 0, 0
    for _ in range(20):
        s = rng.integers(0, 2, PKT).astype(np.int32)
        t = np.concatenate([s, np.zeros(info500['memory'], dtype=np.int32)])
        c = encode(t, info500['coding_length'], info500['polynomials'])
        r = transmit(bpsk_mod(c, 0.5), ch['noise_std'][0])
        L = bpsk_demod(r, ch['noise_var'][0], 0.5)
        _, e = decode(L.astype(np.float32), info500['coding_length'], PKT,
                       info500['n_outputs'], info500['memory'], info500['total_states'],
                       info500['next_states'], info500['outputs'])
        errs += int(np.sum(e != s)); bits += PKT
    ber.append(errs / bits)
bound = ber_coded_union_bound(eb_no_db, d_free=5, rate=0.5)
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(eb_no_db, np.clip(ber, 1e-6, 1), 'o-', label='simulation')
ax.semilogy(eb_no_db, bound, 'k--', label='union bound (d_free=5)')
ax.set_xlabel('Eb/N0 (dB)'); ax.set_ylabel('BER'); ax.set_ylim(1e-6, 1)
ax.grid(True, which='both', alpha=0.3); ax.legend()